> 身份(role) 的定义和理解代入；
- Llama-3.1：base vs Instruct
    - Instruct：有官方 chat template，强依赖它
    - Base：通常没有 chat_template，要自己写
- Qwen2.5-7B：base / Instruct 都是 ChatML，但推荐只对 instruct 聊天用
    - **We do not recommend using base language models for conversations. Instead, you can apply post-training, e.g., SFT, RLHF, continued pretraining, etc., on this model.**
        - https://huggingface.co/Qwen/Qwen2.5-7B
    - base 也能像 instruct model 一样，比较自如地对话
        - 会有不稳定性，未充分 align；

### qwen base

In [28]:
SPECIAL_TOKENS = ["<|endoftext|>", "<|im_start|>", "<|im_end|>"]

In [10]:
import os
from typing import Dict, List

import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [16]:
qwen_base = AutoModelForCausalLM.from_pretrained(
        '/data/zch/weights/Qwen2.5-7B',
        trust_remote_code=True,
        torch_dtype=torch.float16,
    )
qwen_base.eval()
T = AutoTokenizer.from_pretrained('/data/zch/weights/Qwen2.5-7B/')

Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  3.75it/s]


In [19]:
embed = qwen_base.model.embed_tokens.weight.detach().cpu()

In [18]:
T.convert_tokens_to_ids("<|im_start|>"), T.convert_tokens_to_ids("<|im_end|>")

(151644, 151645)

In [30]:
torch.allclose(embed[151644], torch.zeros_like(embed[151644])), torch.allclose(embed[151645], torch.zeros_like(embed[151645]))

(True, True)

In [43]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
analyze_qwen25_special_tokens.py

对 Qwen2.5 Base / Instruct 的 <|im_start|>, <|im_end|> 做 embedding 分析：
- L2 范数
- 邻居 tokens（余弦相似度）
- PCA 2D 可视化
"""

SPECIAL_TOKENS = ["<|endoftext|>", "<|im_start|>", "<|im_end|>"]

def get_special_ids(tokenizer) -> Dict[str, int]:
    ids = {}
    for tok in SPECIAL_TOKENS:
        tid = tokenizer.convert_tokens_to_ids(tok)
        if tid == tokenizer.unk_token_id:
            print(f"[WARN] {tok} 不在 vocab 中，返回 unk id={tid}")
        ids[tok] = tid
    return ids


def load_hf_model(model_name: str, device: str = "cuda"):
    print(f"[INFO] loading model: {model_name}")
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto" if device == "cuda" else None,
    )
    model.eval()
    # Qwen 系列一般是 model.model.embed_tokens
    embed_weight = model.model.embed_tokens.weight.detach().cpu()
    return model, embed_weight


def analyze_norms(embed_weight: torch.Tensor, special_ids: Dict[str, int]):
    norms = embed_weight.norm(dim=-1)  # [V]
    mean = norms.mean().item()
    std = norms.std().item()
    print(f"[STATS] 全 vocab L2 范数: mean={mean:.4f}, std={std:.4f}")

    for tok, tid in special_ids.items():
        n = norms[tid].item()
        z = (n - mean) / (std + 1e-8)
        print(f"  - {tok:12s} (id={tid:6d}): norm={n:.4f}, z-score={z:.2f}")


def print_neighbours(
    embed_weight: torch.Tensor,
    tokenizer,
    special_ids: Dict[str, int],
    k: int = 5,
):
    # 归一化
    # 拷一份，避免改原来的 tensor
    emb = embed_weight.clone()

    # 1. 计算每个 token 的 L2 范数
    norms = emb.norm(dim=-1, keepdim=True)  # [V, 1]
    # 认为 < 1e-6 的就是“零向量”（或极小，容易 0/0）
    zero_mask = norms.squeeze(-1) < 1e-6    # [V]

    # 2. 避免 0/0：对零向量行，把 norm 当成 1 来除（不会改变 0，避免 NaN）
    safe_norms = norms.clone()
    safe_norms[zero_mask] = 1.0
    emb = emb / safe_norms                 # 不会产生 NaN

    for tok, tid in special_ids.items():
        v = emb[tid]  # [d]
        sims = torch.matmul(emb, v)  # [V]
        sims[tid] = -1  # 去掉自己
        topk = torch.topk(sims, k=k)
        ids = topk.indices.tolist()
        vals = topk.values.tolist()
        decoded = [tokenizer.convert_ids_to_tokens(i) for i in ids]
        print(f"\n[NEIGHBORS] {tok} (id={tid}) 最相似的 {k} 个 token：")
        for i, (iid, s, t) in enumerate(zip(ids, vals, decoded)):
            print(f"  {i+1:2d}. id={iid:6d}, sim={s:.4f}, token={t}")

def run_for_model(model_name: str, tag: str, device: str = "cuda"):
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    special_ids = get_special_ids(tokenizer)
    _, embed_weight = load_hf_model(model_name, device=device)

    print(f"\n===== {tag}: Norm 分析 =====")
    analyze_norms(embed_weight, special_ids)

    print(f"\n===== {tag}: 邻居分析 =====")
    print_neighbours(embed_weight, tokenizer, special_ids)


In [44]:
run_for_model('/data/zch/weights/Qwen2.5-7B', "Qwen2.5-7B-Base", device='cuda:2')

[INFO] loading model: /data/zch/weights/Qwen2.5-7B


Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  3.38it/s]



===== Qwen2.5-7B-Base: Norm 分析 =====
[STATS] 全 vocab L2 范数: mean=0.7915, std=0.2046
  - <|endoftext|> (id=151643): norm=0.9746, z-score=0.89
  - <|im_start|> (id=151644): norm=0.0000, z-score=-3.87
  - <|im_end|>   (id=151645): norm=0.0000, z-score=-3.87

===== Qwen2.5-7B-Base: 邻居分析 =====

[NEIGHBORS] <|endoftext|> (id=151643) 最相似的 5 个 token：
   1. id=  4192, sim=0.2423, token=.ĊĊĊ
   2. id=   382, sim=0.2264, token=.ĊĊ
   3. id= 34583, sim=0.2256, token=ĊĊĊĊĊĊĊ
   4. id= 85545, sim=0.2227, token=.ĊĊĊĊĊĊĊĊĊĊĊĊ
   5. id= 24616, sim=0.2170, token=ĊĊĊĊĊĊĊĊĊĊĊĊĊĊĊĊ

[NEIGHBORS] <|im_start|> (id=151644) 最相似的 5 个 token：
   1. id=     3, sim=0.0000, token=$
   2. id=     4, sim=0.0000, token=%
   3. id=     1, sim=0.0000, token="
   4. id=     0, sim=0.0000, token=!
   5. id=     2, sim=0.0000, token=#

[NEIGHBORS] <|im_end|> (id=151645) 最相似的 5 个 token：
   1. id=     3, sim=0.0000, token=$
   2. id=     4, sim=0.0000, token=%
   3. id=     1, sim=0.0000, token="
   4. id=     0, sim=0.0000, 

In [45]:
run_for_model('Qwen/Qwen2.5-7B-Instruct', "Qwen2.5-7B-Instruct", device='cuda:2')

[INFO] loading model: Qwen/Qwen2.5-7B-Instruct


Loading checkpoint shards: 100%|██████████████████████████████████████████████████████████████████| 4/4 [00:01<00:00,  3.63it/s]



===== Qwen2.5-7B-Instruct: Norm 分析 =====
[STATS] 全 vocab L2 范数: mean=0.7905, std=0.2043
  - <|endoftext|> (id=151643): norm=0.9746, z-score=0.90
  - <|im_start|> (id=151644): norm=0.0100, z-score=-3.82
  - <|im_end|>   (id=151645): norm=0.0105, z-score=-3.82

===== Qwen2.5-7B-Instruct: 邻居分析 =====

[NEIGHBORS] <|endoftext|> (id=151643) 最相似的 5 个 token：
   1. id=  4192, sim=0.2407, token=.ĊĊĊ
   2. id=   382, sim=0.2268, token=.ĊĊ
   3. id= 34583, sim=0.2258, token=ĊĊĊĊĊĊĊ
   4. id= 85545, sim=0.2229, token=.ĊĊĊĊĊĊĊĊĊĊĊĊ
   5. id= 24616, sim=0.2157, token=ĊĊĊĊĊĊĊĊĊĊĊĊĊĊĊĊ

[NEIGHBORS] <|im_start|> (id=151644) 最相似的 5 个 token：
   1. id=123818, sim=0.1926, token=×Ł
   2. id=144057, sim=0.1871, token=ï¾Ŀ
   3. id=147896, sim=0.1740, token=ëĪ±
   4. id=150857, sim=0.1705, token=ê´Ļ
   5. id=149726, sim=0.1650, token=ê²Ī

[NEIGHBORS] <|im_end|> (id=151645) 最相似的 5 个 token：
   1. id=123761, sim=0.1848, token=ð¤«ī
   2. id= 13063, sim=0.1782, token=ĉĠĠĠĠĠĠĠĠĠĠĠ
   3. id=148083, sim=0.1735, token=

### qwen contamination

- https://arxiv.org/pdf/2507.10532v1
- https://huggingface.co/datasets/HuggingFaceH4/MATH-500
```
---
For how many positive integers $n>1$ is
---

Generation:
---
 it true that $2^{24}$ is a perfect $n^{\text{th}}$ power?
To determine for how many positive integers \( n > 1 \) the number \( 2^{24} \) is a perfect \( n \)-th power, we need to find the values of \( n \) such that \( 2^{24} = k^n \) for some integer \( k \). This means that \( 24 \) must be divisible by \( n \).

First, we need to find the divisors of \( 24 \). The prime factorization of \( 24 \) is \( 24 = 2^3 \times 3 \). The divisors of \( 24 \) are \( 1, 2, 3, 4, 6, 8, 12, 24 \). Since \( n > 1 \), we exclude \( 1 \) from the list of divisors. Therefore, the possible values for \( n \) are \( 2, 3, 4, 6, 8, 12, 24 \).

Let's count these values. There are \( 7 \) such values. So, the number of positive integers \( n > 1 \) for which \( 2^{24} \) is a perfect \( n \)-th power is \( 7 \).

The final answer is \(\boxed{7}\).
---

finish_reason: stop
```

```python
from vllm import LLM, SamplingParams

# for base model (don't apply (chat) template)
model_id = "Qwen/Qwen2.5-7B"

prompts = [
    # math-500
    "For how many positive integers $n>1$ is",
    "Convert the point $(0,3)$ in rectangular coordinates",
    # gsm8k
    "Josh decides to try flipping a house. He buys a house for $80,000 and then puts"
]

# 3. 根据 Table 1 的 "Greedy (w/o Template)" 配置设置采样参数
# 在 vLLM 中，通过设置 temperature=0 来实现。
# top_p 和 top_k 在贪心解码中不起作用，但我们按惯例设置。
sampling_params = SamplingParams(
    temperature=0,
    top_p=1.0,
    top_k=-1,
    max_tokens=1024,
)

print("="*50)
llm = LLM(model=model_id, max_model_len=4096, gpu_memory_utilization=0.8, trust_remote_code=True)
outputs = llm.generate(prompts, sampling_params)
for i, output in enumerate(outputs):
    original_prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt(partial prompt prefix):\n---\n{original_prompt}\n---\n")
    print(f"Generation:\n---\n{generated_text}\n---\n")
    print(f"finish_reason: {output.outputs[0].finish_reason}")
    print("-"*50)
    
del llm
```